# Chapter 11 — Money, Banking and Monetary Policy

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch11_money_and_monetary_policy.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 11.1 How Indian monetary policy works

## 11.2 Reading an RBI policy decision from the web

In [ ]:
import html
import re
from pathlib import Path

import requests

RAW = Path("data/raw")
HEADERS = {"User-Agent": "python-for-indian-economics (teaching use)"}


def rbi_text(url, name):
    """Plain text of an RBI web page, cached in data/raw/rbi_<name>.txt."""
    path = RAW / f"rbi_{name}.txt"
    if path.exists():
        return path.read_text(encoding="utf-8")
    reply = requests.get(url, headers=HEADERS, timeout=60)
    reply.raise_for_status()
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", reply.text,
                  flags=re.S | re.I)              # drop code blocks first
    text = re.sub(r"<[^>]+>", " ", text)           # then every HTML tag
    text = re.sub(r"\s+", " ", html.unescape(text))
    RAW.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return text


PR = "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid="
first_mpc = rbi_text(PR + "38224", "mpc_2016_10")    # first MPC, Oct 2016
latest_mpc = rbi_text(PR + "63742", "mpc_2026_10")   # MPC, October 2026
print(len(first_mpc), "and", len(latest_mpc), "characters of text")

In [ ]:
decision = re.compile(r"(reduce|increase)\w* the policy repo rate.*?"
                      r"by (\d+) (?:bps|basis points).*?to (\d+\.\d+) per cent")

for label, text in [("Oct 2016", first_mpc), ("Oct 2026", latest_mpc)]:
    m = decision.search(text)
    print(f"{label}: {m.group(1)} the repo rate by {m.group(2)} basis points "
          f"to {m.group(3)} per cent")

In [ ]:
def find(pattern, text):
    m = re.search(pattern, text)
    return m.group(1) if m else None


print("Date of release:     ", find(r"Date : (\w{3} \d{2}, \d{4})", latest_mpc))
print("Meeting number:      ", find(r"held its (\d+)\w\w meeting", latest_mpc))
print("SDF rate (floor):    ",
      find(r"\(SDF\) rate stands adjusted at (\d+\.\d+)", latest_mpc))
print("MSF rate (ceiling):  ",
      find(r"\(MSF\) rate and the Bank Rate at (\d+\.\d+)", latest_mpc))
cpi_proj = find(r"CPI inflation is projected to be (\d+\.\d+) per cent for 2026-27",
                latest_mpc)
print("CPI projection 2026-27:", cpi_proj)

## 11.3 The data: three interest rates on one calendar

In [ ]:
import numpy as np
import pandas as pd
from indiadata import bis, oecd_rates, mospi_cpi

repo = bis("WS_CBPOL", "M.IN").rename(columns={"value": "repo"})
market = oecd_rates()
print("Repo rate:   ", repo["date"].min().date(), "to", repo["date"].max().date())
print("OECD rates:  ", market["date"].min().date(), "to", market["date"].max().date())
market.tail(3)

In [ ]:
oct_decision = pd.Timestamp("2026-10-07")
month = oct_decision.to_period("M").to_timestamp()
print(oct_decision.date(), "belongs to the month starting", month.date())

latest = pd.DataFrame({"date": [month],
                       "repo": [float(decision.search(latest_mpc).group(3))]})
if month not in set(repo["date"]):          # only if BIS has not caught up
    repo = pd.concat([repo, latest], ignore_index=True)
repo.tail(3)

In [ ]:
rates = (repo.merge(market, on="date", how="inner")
             .set_index("date")
             .sort_index())
print(len(rates), "months:", rates.index.min().date(), "to",
      rates.index.max().date())
rates.describe().round(2)

## 11.4 A first look

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE, LIGHT = "#1f4e79", "#c55a11", "#9db7d5"

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.step(rates.index, rates["repo"], where="post", color=BLUE, linewidth=2.2,
        label="Policy repo rate")
ax.plot(rates.index, rates["short_rate"], color=ORANGE, linewidth=1.4,
        label="3-month interbank rate")
ax.plot(rates.index, rates["long_rate"], color="grey", linewidth=1.4,
        label="10-year G-sec yield")
ax.set_title("India: the repo rate and market interest rates", loc="left",
             fontsize=13)
ax.set_ylabel("% per year")
ax.legend(loc="upper right", frameon=False, ncol=3, fontsize=9)
ax.set_ylim(2.5, 13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Sources: BIS central bank policy rates; OECD Financial "
         "Market Statistics; RBI press release (Oct 2026). Monthly.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
rates[["repo", "short_rate", "long_rate"]].corr().round(2)

## 11.5 Policy-rate cycles

In [ ]:
rates["d_repo"] = rates["repo"].diff()
moves = rates.loc[rates["d_repo"].fillna(0) != 0, ["repo", "d_repo"]].copy()
moves["bp"] = (moves["d_repo"] * 100).round().astype(int)   # basis points
print("Number of months with a repo change:", len(moves))
print("  hikes:", (moves["bp"] > 0).sum(), "  cuts:", (moves["bp"] < 0).sum())
moves.tail(8)

In [ ]:
moves["sign"] = np.sign(moves["bp"])
moves["cycle"] = (moves["sign"] != moves["sign"].shift()).cumsum()

cycles = (moves.reset_index()
               .groupby("cycle")
               .agg(start=("date", "min"), end=("date", "max"),
                    moves=("bp", "size"), total_bp=("bp", "sum")))
cycles["type"] = np.where(cycles["total_bp"] > 0, "hiking", "easing")
cycles

## 11.6 Spreads: how far do market rates sit from the repo rate?

In [ ]:
rates["short_spread"] = rates["short_rate"] - rates["repo"]
rates["term_spread"] = rates["long_rate"] - rates["repo"]
rates[["short_spread", "term_spread"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axhline(0, color="black", linewidth=0.8)
ax.plot(rates.index, rates["short_spread"], color=ORANGE, linewidth=1.5,
        label="3-month rate minus repo")
ax.plot(rates.index, rates["term_spread"], color="grey", linewidth=1.5,
        label="10-year yield minus repo")
ax.set_title("Spreads of market rates over the repo rate", loc="left",
             fontsize=13)
ax.set_ylabel("Percentage points")
ax.annotate("July–Sept 2013:\nliquidity tightened\nto defend the rupee",
            xy=(pd.Timestamp("2013-08-01"), 4.0),
            xytext=(pd.Timestamp("2015-01-01"), 3.3), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="grey"))
ax.legend(loc="upper right", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Sources: BIS; OECD. Spread = market rate (monthly "
         "average) minus repo rate (end of month).", fontsize=8, color="grey")
plt.show()

In [ ]:
chron = rbi_text("https://www.rbi.org.in/scripts/PublicationsView.aspx?id=15544",
                 "chronology_2013")
for m in re.finditer(r"MSF rate (?:recalibrated|reduced by 75).*?\d{4}\.", chron):
    print("*", m.group(0))

## 11.7 Mixed frequencies: from months to quarters and fiscal years

In [ ]:
quarterly = rates[["repo", "short_rate", "long_rate"]].resample("QE").mean()
quarterly.tail(4).round(2)

In [ ]:
fiscal = rates[["repo", "short_rate", "long_rate"]].resample("YE-MAR").mean()
fiscal.index = [f"{d.year - 1}-{str(d.year)[2:]}" for d in fiscal.index]
fiscal.round(2).tail(6)

## 11.8 How fast? Lags with `shift`

In [ ]:
d = rates[["repo", "short_rate", "long_rate"]].diff()

lag_table = pd.DataFrame({
    "3-month rate": [d["short_rate"].corr(d["repo"].shift(k)) for k in range(7)],
    "10-year yield": [d["long_rate"].corr(d["repo"].shift(k)) for k in range(7)],
}, index=pd.Index(range(7), name="lag (months)"))
lag_table.round(2)

## 11.9 A pass-through regression

In [ ]:
import statsmodels.formula.api as smf

reg = pd.DataFrame({"d_short": d["short_rate"], "d_long": d["long_rate"]})
for k in range(4):
    reg[f"d_repo_L{k}"] = d["repo"].shift(k)
rhs = " + ".join(f"d_repo_L{k}" for k in range(4))

results = {}
for y in ["d_short", "d_long"]:
    results[y] = smf.ols(f"{y} ~ {rhs}", data=reg).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
print(results["d_short"].summary().tables[1])

In [ ]:
summary = []
for y, name in [("d_short", "3-month rate"), ("d_long", "10-year yield")]:
    res = results[y]
    total = res.t_test(" + ".join(f"d_repo_L{k}" for k in range(4)) + " = 0")
    summary.append({"market rate": name,
                    "impact (lag 0)": res.params["d_repo_L0"],
                    "cumulative, 3 months": float(total.effect[0]),
                    "s.e. of cumulative": float(total.sd[0][0]),
                    "months used": int(res.nobs)})
pd.DataFrame(summary).set_index("market rate").round(2)

## 11.10 Event windows: what happens after a repo move?

In [ ]:
def event_window(series, date, before=3, after=9):
    """Change in `series` relative to the month before `date`."""
    i = rates.index.get_loc(date)
    if i - before < 0 or i + after >= len(rates):
        return None                        # window runs off the data
    path = series.iloc[i - before: i + after + 1].to_numpy() - series.iloc[i - 1]
    return pd.Series(path, index=range(-before, after + 1))


events = moves[moves.index.year != 2013]      # leave out the 2013 episode
paths = {"repo": [], "short_rate": [], "long_rate": []}
for date, row in events.iterrows():
    for col in paths:
        w = event_window(rates[col], date)
        if w is not None:
            paths[col].append(w * row["sign"])     # flip cuts into "hikes"
avg = pd.DataFrame({col: pd.concat(p, axis=1).mean(axis=1)
                    for col, p in paths.items()})
print("Events used:", len(paths["repo"]))
avg.loc[[-1, 0, 1, 3, 6, 9]].round(2)

In [ ]:
ratio = pd.DataFrame({"3-month rate": avg["short_rate"] / avg["repo"],
                      "10-year yield": avg["long_rate"] / avg["repo"]})
ratio.loc[[0, 1, 3, 6, 9]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
ax.step(avg.index, avg["repo"], where="post", color=BLUE, linewidth=2.2,
        label="Policy repo rate")
ax.plot(avg.index, avg["short_rate"], marker="o", color=ORANGE,
        label="3-month interbank rate")
ax.plot(avg.index, avg["long_rate"], marker="o", color="grey",
        label="10-year G-sec yield")
ax.set_xticks(range(-3, 10))
ax.set_xlabel("Months from the repo change (0 = month of the decision)")
ax.set_ylabel("Change since month −1, % points")
ax.set_title("Average path of rates around a repo-rate move", loc="left",
             fontsize=13)
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.04, "Repo changes 2012–2025, excluding 2013; cuts multiplied "
         "by −1 so that all moves point up. Sources: BIS; OECD.",
         fontsize=8, color="grey")
plt.show()

## 11.11 Transmission over whole cycles

In [ ]:
def cycle_change(col, start, end):
    before = rates[col].loc[:start].iloc[-2]              # month before start
    after = rates[col].loc[end:].iloc[1:4]                # 3 months after end
    return after.mean() - before if len(after) == 3 else np.nan


table = cycles[["type", "start", "end", "total_bp"]].copy()
table["repo"] = table["total_bp"] / 100
table["3-month"] = [cycle_change("short_rate", s, e)
                    for s, e in zip(cycles["start"], cycles["end"])]
table["10-year"] = [cycle_change("long_rate", s, e)
                    for s, e in zip(cycles["start"], cycles["end"])]
table["ratio 3m"] = table["3-month"] / table["repo"]
table["ratio 10y"] = table["10-year"] / table["repo"]
table["start"] = table["start"].dt.strftime("%b %Y")
table["end"] = table["end"].dt.strftime("%b %Y")
table.drop(columns="total_bp").round(2)

## 11.12 The real policy rate

In [ ]:
cpi = mospi_cpi("combined")[["date", "inflation"]].set_index("date")
rates = rates.join(cpi, how="left")          # keep every month of rates
rates["real_repo"] = rates["repo"] - rates["inflation"]
print("CPI inflation available for",
      rates["inflation"].notna().sum(), "of", len(rates), "months; last:",
      rates["inflation"].last_valid_index().date())

In [ ]:
real_fy = rates["real_repo"].resample("YE-MAR").agg(["mean", "count"])
real_fy.index = [f"{d.year - 1}-{str(d.year)[2:]}" for d in real_fy.index]
real_fy = real_fy[real_fy["count"] >= 10]            # nearly full years only
print("Highest monthly real rate:", round(rates["real_repo"].max(), 2), "in",
      rates["real_repo"].idxmax().strftime("%B %Y"))
print("Lowest monthly real rate: ", round(rates["real_repo"].min(), 2), "in",
      rates["real_repo"].idxmin().strftime("%B %Y"))
real_fy.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.3))
r = rates["real_repo"]
ax.axhline(0, color="black", linewidth=0.8)
ax.fill_between(r.index, r, 0, where=r >= 0, color=LIGHT, interpolate=True,
                label="positive")
ax.fill_between(r.index, r, 0, where=r < 0, color=ORANGE, alpha=0.6,
                interpolate=True, label="negative")
ax.plot(r.index, r, color=BLUE, linewidth=1.3)
ax.set_title("India: the real (ex post) repo rate", loc="left", fontsize=13)
ax.set_ylabel("Repo minus CPI inflation, % points")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Sources: BIS policy rate; MoSPI CPI-Combined, "
         "year-on-year. Gap = April–May 2020, when no inflation was published.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
ex_ante = float(decision.search(latest_mpc).group(3)) - float(cpi_proj)
print(f"Repo (Oct 2026) minus projected CPI inflation for 2026-27: "
      f"{ex_ante:.2f} percentage points")

## 11.13 From market rates to bank lending rates

In [ ]:
ebl = rbi_text("https://www.rbi.org.in/Scripts/NotificationUser.aspx?Id=11677&Mode=0",
               "circular_ebl_2019")
for pat in [r"The ISG observed that[^.]*\.",
            r"extended by banks from October 01, 2019 shall be benchmarked to[^:]*:",
            r"The interest rate under external benchmark shall be reset[^.]*\."]:
    print("*", re.search(pat, ebl).group(0))

In [ ]:
walr_file = Path("data/manual/rbi_walr.csv")
if walr_file.exists():
    walr = pd.read_csv(walr_file, parse_dates=["date"]).set_index("date")
    both = rates.join(walr, how="inner")
    print("Months matched:", len(both))
    print("Correlation of monthly changes with repo, lags 0-6:")
    dw = both[["repo", "walr_fresh"]].diff()
    print([round(dw["walr_fresh"].corr(dw["repo"].shift(k)), 2) for k in range(7)])
else:
    print("No file at", walr_file, "- skipping the bank lending-rate analysis.")
    print("See the instructions above to add it.")

## 11.14 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading